In [ ]:
import pandas as pd
import numpy as np
b = pd.read_csv("bank_transactions.csv", parse_dates=["date"])
ar = pd.read_csv("accounts_receivable.csv", parse_dates=["invoice_date", "due_date", "paid_date"])
ap = pd.read_csv("accounts_payable.csv", parse_dates=["bill_date", "due_date"])
debt = pd.read_csv("debt_facilities.csv")
op = pd.read_csv("operating_forecast.csv")
W1 = pd.Timestamp("2026-11-23")
r = debt.iloc[0]
W = lambda s: (s - W1).dt.days.clip(0) // 7 + 1
c = lambda x: cash0 + x.groupby("week").amount.sum().loc[:13].cumsum()

def export():
    pos.to_csv("reconciled_cash_position.csv")
    fc.to_csv("13_week_liquidity_forecast.csv")
    bba.to_csv("borrowing_base_analysis.csv")
    t.to_csv("scenario_comparison.csv")
    k.to_csv("liquidity_risk_report.csv")

for n, d in {"bank": b, "ar": ar, "ap": ap, "debt": debt, "op": op}.items():
    print(n, d.shape, "| duplicate rows:", d.duplicated().sum())
debt.set_index("facility_id").T

In [ ]:
pos = b.drop_duplicates("txn_id").groupby(["account_type", "status"]).amount.sum()
cash0 = pos["operating"].sum()
pos

In [ ]:
ar["late"] = (ar.paid_date - ar.due_date).dt.days
d = ar[ar.paid_date > "2026-08-22"].groupby("customer").late.median()
d

In [ ]:
o = ar[ar.status.eq("open")].assign(week=W(ar.due_date + pd.to_timedelta(ar.customer.map(d), "D")))
f = pd.concat([o, ap.assign(week=W(ap.due_date), amount=-ap.amount), op.assign(week=op.week + op.customer.map(d).fillna(0) // 7)])
fc = f.groupby(["week", f.amount > 0]).amount.sum().unstack().set_axis(["outflow", "inflow"], axis=1).loc[:13].assign(close=c(f))
fc

In [ ]:
e = o.groupby("customer").amount.sum().clip(upper=o.amount.sum() / 4).sum()
bba = pd.Series({"eligible": e, "base": r.advance_rate * e - r.reserve - r.drawn, "commitment": r.commitment - r.drawn, "leverage": r.max_leverage * r.ltm_ebitda - debt.drawn.sum()})
cap = bba.iloc[1:].min()
bba

In [ ]:
p = f[f.discretionary.ne("Y")]
q = p.assign(week=p.week + p.max_extension_days.fillna(0) // 7)
N = lambda x: x.assign(week=x.week + 2 * x.customer.eq(d.idxmax()))
x = np.ceil((500000 - c(N(q)).min()) / 25000) * 25000
t = pd.DataFrame({"base": c(f), "credit": c(f) + cap, "credit_stress": c(N(f)) + cap, "levers": c(q), "combined": c(q) + x, "stress": c(N(q)) + x})
t.plot().axhline(500000);

In [ ]:
k = pd.Series({"draw": x, "leverage": (debt.drawn.sum() + x) / r.ltm_ebitda})
export()
print(k)
t.min()